# NB20b — do the occlusion re-fits reproduce the stored runs? (CPU; rebuilds the cache from OpenNeuro)

Downloads ds004504, runs the fixed preamble and feature cache for all 88 subjects (as NB01 + NB02, unsharded), then re-fits P3 x spatial on the 25 selection splits with the main-grid seeds and compares the subject probabilities with the stored runs. Small differences can also come from library versions.

In [ ]:
REPO = 'https://github.com/Devguru-codes/Major_project_IIITN.git'
BRANCH = 'feat/implementation'
WORK = '/kaggle/working'
SRC = '/tmp/eegrep-src'   # code checkout stays out of the saved output

import subprocess, sys, os, json, platform, shutil

def sh(cmd, log=None):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-20000:], r.stderr[-5000:], sep='\n', flush=True)
    if log:
        open(log, 'w').write(r.stdout + '\n' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.stdout

shutil.rmtree(SRC, ignore_errors=True)
sh(f'git clone --depth 1 -b {BRANCH} {REPO} {SRC}')
SHA = sh(f'git -C {SRC} rev-parse --short HEAD').strip()
sh(f'pip install -q -e "{SRC}[preprocess,dev,stats]"', log=f'{WORK}/pip_install.log')
PY = f'cd {SRC} && {sys.executable} -m eegrep'
print('git sha', SHA)

In [ ]:
sh(f'{PY} download --dest /tmp/ds004504', log=f'{WORK}/download.log')
shutil.copy('/tmp/ds004504/participants.tsv', f'{WORK}/participants.tsv')

In [ ]:
sh(f'{PY} preprocess --bids /tmp/ds004504 --out /tmp/preproc --n-jobs 4', log=f'{WORK}/preprocess.log')

In [ ]:
sh(f'{PY} cache --preproc /tmp/preproc --participants {WORK}/participants.tsv --out /tmp/cache', log=f'{WORK}/cache.log')

In [ ]:
sh(f'{PY} refit-check --cache /tmp/cache --folds {SRC}/splits/folds_ds004504.json --runs {SRC}/reports/grid/runs_s0.jsonl {SRC}/reports/grid/runs_s1.jsonl {SRC}/reports/grid/runs_s2.jsonl --tag main --pipeline P3 --edge spatial --out {WORK}/refit', log=f'{WORK}/refit.log')

In [ ]:
import torch
sh(f'{sys.executable} -m pip freeze', log=f'{WORK}/pip_freeze.txt')
env = {'git_sha': SHA, 'python': platform.python_version(), 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(), 'cpu_count': os.cpu_count(),
       'kaggle_image': os.environ.get('KAGGLE_DOCKER_IMAGE', 'unknown')}
json.dump(env, open(f'{WORK}/environment.json', 'w'), indent=1)
print(env)